# 📘 Project 01 — Electronic Health Record Analysis
**Team No.:** 14  **Team Members:** Prativa Priyadarshini Behera; Samikhya Pati; Prakruti Das; Sandeep Behera

**Proposed Hybrid Model:** Temporal Transformer + Heterogeneous Patient GAT

**Dataset / Source:** Diabetes 130-US hospitals for years 1999–2008 (Kaggle mirror; tracker/README label it "MIMIC-IV derived extract" — this is inaccurate, see Data-Model Compatibility Note below)
**Dataset Link:** https://www.kaggle.com/datasets/brandao/diabetes

**Task Type:** Binary classification — early hospital readmission (`readmitted < 30 days`)

---
## Data-Model Compatibility Note (read before running)
The tracker/README describe this dataset as a *"MIMIC-IV derived extract"*, but the linked Kaggle source is actually the **Diabetes 130-US hospitals (1999–2008)** dataset (Strack et al., 2014) — a flat encounter table, not a MIMIC-IV ICU extract. This is a tracker/README labeling error, not a modeling choice; the actual downloadable data is the diabetes-130 encounter table, so that is what this notebook uses.

The proposed model (**Temporal Transformer + Heterogeneous Patient GAT**) needs (a) a temporal signal and (b) a non-trivial patient graph. Neither exists off-the-shelf in this dataset, but both can be **honestly derived from columns that are actually present** — no modality is fabricated:

- **Temporal branch**: `patient_nbr` repeats for patients with multiple hospital encounters. Grouping by `patient_nbr` and ordering by `encounter_id` (monotonically increasing = chronological in this dataset) gives a genuine, if short, per-patient encounter sequence. Single-encounter patients get a sequence of length 1 — the Transformer handles that natively via padding + a mask.
- **Heterogeneous graph branch**: `diag_1` (primary ICD-9 diagnosis category) gives a real **bipartite patient–diagnosis graph** (two genuine node types: patient nodes, diagnosis nodes; edges = "patient was diagnosed with X"). This is an authentic heterogeneous structure already implicit in the data, not a synthetic similarity graph.
- **Target**: `readmitted` is a 3-way field (`NO` / `>30` / `<30`). Following the standard formulation in the diabetes-130 readmission literature (Strack et al. 2014; Beata Strack's own paper's headline task), it is binarized to `readmitted_lt30` — predicting the clinically actionable early-readmission event.

**Verdict: PARTIAL compatibility, adaptation documented above.** No modality is invented; both branches use only columns present in the actual downloaded file.

**How to run:** `Runtime → Change runtime type → GPU`, then `Runtime → Run all`. You will be prompted to upload a Kaggle API token (`kaggle.json`) in Section 1.


## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
# Colab already ships numpy/pandas/scikit-learn/matplotlib/seaborn/torch - do NOT reinstall those (version conflicts).
# Only install what's actually missing.
!pip -q install kaggle shap tqdm tabulate


In [ ]:
import os
import sys
import json
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    matthews_corrcoef
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device selected:", DEVICE)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


### CONFIG — project-specific settings

In [ ]:
CONFIG = {
    "project_no": "01",
    "project_name": "Electronic_Health_Record_Analysis",
    "team_no": "14",
    "task_type": "classification",
    "modality": "tabular_temporal_graph",
    "kaggle_dataset_slug": "brandao/diabetes",
    "dataset_source": "Diabetes 130-US hospitals for years 1999-2008 (Strack et al., 2014)",
    "target_column": "readmitted_lt30",
    "raw_target_column": "readmitted",
    "id_columns": ["patient_nbr"],
    "time_column": None,  # no real timestamp; encounter_id ordering used instead, handled explicitly in Section 5
    "diagnosis_column": "diag_1",
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "max_seq_len": 5,       # cap per-patient encounter sequence length (temporal branch)
    "gat_hidden_dim": 32,
    "transformer_hidden_dim": 64,
    "transformer_heads": 4,
    "batch_size": 128,
    "epochs": 25,
    "learning_rate": 1e-3,
    "early_stop_patience": 5,
    "data_raw_dir": "data/raw",
    "data_processed_dir": "data/processed",
    "figures_dir": "figures",
    "results_dir": "results",
    "reports_dir": "reports",
}

for d in [CONFIG["data_raw_dir"], CONFIG["data_processed_dir"], CONFIG["figures_dir"],
          CONFIG["results_dir"], CONFIG["reports_dir"]]:
    os.makedirs(d, exist_ok=True)

CONFIG


## 1. Dataset Download
Programmatic download via the Kaggle API — no manual clicking, so the notebook is reproducible end to end.

In [ ]:
# --- Kaggle API download ---
# 1) Get an API token from https://www.kaggle.com/settings -> "Create New Token" (downloads kaggle.json)
# 2) Run this cell and upload it when prompted

from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        if fname.endswith(".json"):
            os.replace(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d {CONFIG["kaggle_dataset_slug"]} -p {CONFIG["data_raw_dir"]} --unzip


In [ ]:
# --- Sanity check the download ---
raw_files = os.listdir(CONFIG["data_raw_dir"])
print("Files in raw data dir:", raw_files)
assert len(raw_files) > 0, "No files found - check Section 1 download step before continuing."

for f in raw_files:
    p = os.path.join(CONFIG["data_raw_dir"], f)
    print(f, "-", os.path.getsize(p), "bytes")


## 2. Load Raw Data

In [ ]:
# The Kaggle mirror ships diabetic_data.csv (main table) and IDS_mapping.csv (ID lookup tables).
# Identify the main encounter-level file by name rather than assuming a fixed filename.
candidates = [f for f in raw_files if "diabetic" in f.lower() and f.lower().endswith(".csv")]
assert len(candidates) >= 1, f"Could not find the main diabetic_data CSV among: {raw_files}"
RAW_FILE = os.path.join(CONFIG["data_raw_dir"], candidates[0])
print("Using raw file:", RAW_FILE)

df = pd.read_csv(RAW_FILE, na_values=["?"])
print(df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Shape, dtypes, duplicates
print("Shape:", df.shape)
print(df.dtypes.value_counts())
print("Duplicate rows:", df.duplicated().sum())
print("Unique patients:", df["patient_nbr"].nunique(), "/ total encounters:", len(df))
print("Patients with >1 encounter:", (df["patient_nbr"].value_counts() > 1).sum())


In [ ]:
# Missingness
missing = df.isnull().mean().sort_values(ascending=False)
plt.figure(figsize=(8, 6))
missing[missing > 0].plot(kind="barh")
plt.title("Missing value proportion by column")
plt.xlabel("Fraction missing")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_missingness.png"), dpi=300)
plt.show()
print(missing[missing > 0])


In [ ]:
# Build the binarized target and inspect distribution
df["readmitted_lt30"] = (df["readmitted"] == "<30").astype(int)

plt.figure(figsize=(6, 4))
sns.countplot(x=df["readmitted_lt30"])
plt.title("Target distribution: readmitted < 30 days")
plt.xlabel("readmitted_lt30 (1 = readmitted within 30 days)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_target_distribution.png"), dpi=300)
plt.show()

print(df["readmitted"].value_counts())
print(df["readmitted_lt30"].value_counts(normalize=True))


In [ ]:
# Diagnosis-code coverage (needed for the heterogeneous graph branch)
print("Missing diag_1:", df["diag_1"].isnull().mean())
print("Unique diag_1 codes:", df["diag_1"].nunique())
print(df["diag_1"].value_counts().head(10))

# Per-patient encounter sequence length (needed for the temporal branch)
enc_counts = df.groupby("patient_nbr").size()
plt.figure(figsize=(6, 4))
sns.histplot(enc_counts.clip(upper=10), bins=10)
plt.title("Encounters per patient (clipped at 10)")
plt.xlabel("Number of encounters")
plt.tight_layout()
plt.show()
print(enc_counts.describe())


**Data quality memo** — auto-generated from the EDA above, saved to `reports/data_quality_memo.md`.

In [ ]:
top_missing = missing[missing > 0].head(5)
n_multi_encounter_patients = int((enc_counts > 1).sum())

data_quality_memo = f"""# Data Quality Memo - Project 01: Electronic Health Record Analysis

## Dataset
- Source: Diabetes 130-US hospitals for years 1999-2008 (brandao/diabetes on Kaggle)
- Rows (encounters): {len(df)}
- Unique patients: {df['patient_nbr'].nunique()}
- Duplicate rows: {df.duplicated().sum()}

## Target
- readmitted_lt30 positive rate: {df['readmitted_lt30'].mean():.4f}
- Class imbalance present (~{df['readmitted_lt30'].mean()*100:.1f}% positive) - handled via class-weighted loss
  and macro-averaged metrics (Section 9), not by resampling, to keep the test distribution untouched.

## Missingness (top columns)
{top_missing.to_string()}
- `weight`, `payer_code`, `medical_specialty` have very high missingness in this dataset family
  (a known property of the source, not a pipeline bug) - dropped or coarsely bucketed in Section 4.

## Leakage risks identified
- `patient_nbr` repeats across rows ({n_multi_encounter_patients} patients with >1 encounter):
  the same patient's encounters must never be split across train/val/test -> group-aware split
  used in Section 5 (GroupShuffleSplit on patient_nbr).
- `encounter_id` is monotonically increasing and used only as an ordering key for the temporal
  branch, never as a model input feature (it would leak record order / dataset construction).
- `discharge_disposition_id` includes "expired"/hospice codes that make readmission structurally
  impossible for those encounters; these rows are excluded in Section 4 to avoid a label-definition
  leak.

## Adaptation note
- No real MIMIC-IV timestamps or explicit patient-relationship graph exist in this dataset.
  Temporal branch uses per-patient encounter order (encounter_id); graph branch uses a genuine
  bipartite patient-diagnosis (diag_1) structure. See notebook header for full rationale.
"""
with open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"), "w") as f:
    f.write(data_quality_memo)
print(data_quality_memo)


## 4. Preprocessing & Feature Engineering

In [ ]:
# Exclude encounters where readmission is structurally impossible (expired / hospice discharge)
EXPIRED_HOSPICE_CODES = {11, 13, 14, 19, 20, 21}
df = df[~df["discharge_disposition_id"].isin(EXPIRED_HOSPICE_CODES)].reset_index(drop=True)

# Drop columns that are near-fully missing or free-text/administrative and not predictive features
drop_cols = ["weight", "payer_code", "medical_specialty", "readmitted"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Coarsen the three diagnosis columns to their ICD-9 3-digit category (drops decimal detail, keeps signal)
def icd9_category(code):
    if pd.isnull(code):
        return "missing"
    code = str(code)
    if code.startswith(("V", "E")):
        return code[0]
    try:
        return str(int(float(code)) // 100)
    except ValueError:
        return "other"

for c in ["diag_1", "diag_2", "diag_3"]:
    df[c + "_cat"] = df[c].apply(icd9_category)

print(df.shape)
df.head()


## 5. Train / Validation / Test Split

In [ ]:
ratios = CONFIG["split_ratios"]
target = CONFIG["target_column"]

# Group-aware split on patient_nbr: the same patient's encounters never span more than one split.
gss = GroupShuffleSplit(n_splits=1, train_size=ratios["train"], random_state=CONFIG["random_seed"])
groups = df["patient_nbr"]
train_idx, rest_idx = next(gss.split(df, groups=groups))
train_df, rest_df = df.iloc[train_idx].reset_index(drop=True), df.iloc[rest_idx].reset_index(drop=True)

rel_val = ratios["val"] / (ratios["val"] + ratios["test"])
gss2 = GroupShuffleSplit(n_splits=1, train_size=rel_val, random_state=CONFIG["random_seed"])
val_idx, test_idx = next(gss2.split(rest_df, groups=rest_df["patient_nbr"]))
val_df, test_df = rest_df.iloc[val_idx].reset_index(drop=True), rest_df.iloc[test_idx].reset_index(drop=True)

print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))

# Assert leakage prevention: no patient_nbr appears in more than one split
train_p, val_p, test_p = set(train_df.patient_nbr), set(val_df.patient_nbr), set(test_df.patient_nbr)
assert not (train_p & val_p), "Patient leakage between train and val!"
assert not (train_p & test_p), "Patient leakage between train and test!"
assert not (val_p & test_p), "Patient leakage between val and test!"
print("Group-split leakage check passed: no patient appears in more than one split.")

manifest = {
    "train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df),
    "train_class_balance": train_df[target].value_counts(normalize=True).to_dict(),
    "val_class_balance": val_df[target].value_counts(normalize=True).to_dict(),
    "test_class_balance": test_df[target].value_counts(normalize=True).to_dict(),
}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)

train_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "train.csv"), index=False)
val_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "val.csv"), index=False)
test_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "test.csv"), index=False)
manifest


In [ ]:
# Fit preprocessing transformers on TRAIN ONLY, then apply to val/test.
numeric_features = [
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
]
categorical_features = [
    "race", "gender", "age", "admission_type_id", "discharge_disposition_id",
    "admission_source_id", "max_glu_serum", "A1Cresult", "change", "diabetesMed",
    "diag_1_cat", "diag_2_cat", "diag_3_cat",
]

imputer = SimpleImputer(strategy="median").fit(train_df[numeric_features])
scaler = StandardScaler()
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_features] = imputer.transform(split_df[numeric_features])
scaler.fit(train_df[numeric_features])
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_features] = scaler.transform(split_df[numeric_features])

for c in categorical_features:
    train_df[c] = train_df[c].astype(str).fillna("missing")
    val_df[c] = val_df[c].astype(str).fillna("missing")
    test_df[c] = test_df[c].astype(str).fillna("missing")

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_df[categorical_features])
cat_feature_names = encoder.get_feature_names_out(categorical_features).tolist()

def encode_cat(split_df):
    return pd.DataFrame(encoder.transform(split_df[categorical_features]), columns=cat_feature_names, index=split_df.index)

train_cat = encode_cat(train_df)
val_cat = encode_cat(val_df)
test_cat = encode_cat(test_df)

TABULAR_FEATURE_COLS = numeric_features + cat_feature_names
print("Total tabular feature dim:", len(TABULAR_FEATURE_COLS))

train_X = pd.concat([train_df[numeric_features].reset_index(drop=True), train_cat.reset_index(drop=True)], axis=1)
val_X = pd.concat([val_df[numeric_features].reset_index(drop=True), val_cat.reset_index(drop=True)], axis=1)
test_X = pd.concat([test_df[numeric_features].reset_index(drop=True), test_cat.reset_index(drop=True)], axis=1)


In [ ]:
# --- Heterogeneous patient-diagnosis graph (Section 7's GAT branch input) ---
# Real bipartite structure: patient nodes <-> diagnosis-category nodes (from diag_1_cat).
all_diag_cats = sorted(pd.concat([train_df["diag_1_cat"], val_df["diag_1_cat"], test_df["diag_1_cat"]]).unique())
diag_to_idx = {d: i for i, d in enumerate(all_diag_cats)}
N_DIAG_NODES = len(diag_to_idx)
print("Distinct diagnosis-category nodes:", N_DIAG_NODES)

def diag_index(split_df):
    return split_df["diag_1_cat"].map(diag_to_idx).values

train_diag_idx = diag_index(train_df)
val_diag_idx = diag_index(val_df)
test_diag_idx = diag_index(test_df)


## 6. PyTorch Dataset & DataLoader

In [ ]:
class EHRDataset(Dataset):
    """Returns tabular features, per-patient encounter sequence, diagnosis-node index, and target.

    Temporal branch input: for each row, the sequence of that patient's OTHER encounters in this
    split up to CONFIG['max_seq_len'], ordered by encounter_id, zero-padded. This keeps the
    temporal branch leakage-safe (only uses encounters within the same split).
    """
    def __init__(self, tab_df, raw_df, diag_idx, target_col, max_seq_len):
        self.tab = tab_df.reset_index(drop=True).values.astype(np.float32)
        self.raw = raw_df.reset_index(drop=True)
        self.diag_idx = diag_idx
        self.target = raw_df[target_col].values.astype(np.float32)
        self.max_seq_len = max_seq_len
        self.feat_dim = self.tab.shape[1]

        # Precompute per-patient row order (by encounter_id) within this split only
        self.raw = self.raw.assign(_row=np.arange(len(self.raw)))
        self.patient_rows = {
            pid: g.sort_values("encounter_id")["_row"].tolist()
            for pid, g in self.raw.groupby("patient_nbr")
        }

    def __len__(self):
        return len(self.raw)

    def __getitem__(self, idx):
        pid = self.raw.iloc[idx]["patient_nbr"]
        rows = self.patient_rows[pid]
        pos = rows.index(idx)
        seq_rows = rows[max(0, pos - self.max_seq_len + 1): pos + 1]

        seq = np.zeros((self.max_seq_len, self.feat_dim), dtype=np.float32)
        mask = np.zeros((self.max_seq_len,), dtype=np.float32)
        n = len(seq_rows)
        seq[-n:] = self.tab[seq_rows]
        mask[-n:] = 1.0

        x_tab = torch.tensor(self.tab[idx])
        x_seq = torch.tensor(seq)
        x_mask = torch.tensor(mask)
        diag = torch.tensor(int(self.diag_idx[idx]), dtype=torch.long)
        y = torch.tensor(self.target[idx])
        return x_tab, x_seq, x_mask, diag, y


BATCH_SIZE = CONFIG["batch_size"]

train_ds = EHRDataset(train_X, train_df, train_diag_idx, target, CONFIG["max_seq_len"])
val_ds = EHRDataset(val_X, val_df, val_diag_idx, target, CONFIG["max_seq_len"])
test_ds = EHRDataset(test_X, test_df, test_diag_idx, target, CONFIG["max_seq_len"])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Verify one batch
xb_tab, xb_seq, xb_mask, xb_diag, yb = next(iter(train_loader))
print("tabular:", xb_tab.shape, "sequence:", xb_seq.shape, "mask:", xb_mask.shape,
      "diag idx:", xb_diag.shape, "target:", yb.shape)


## 7. Model Definitions

In [ ]:
class TemporalTransformerBranch(nn.Module):
    """Encodes each patient's encounter sequence (Section 6) with a Transformer encoder,
    returning the representation of the current (last, always-unmasked) encounter."""
    def __init__(self, input_dim, hidden_dim=64, n_heads=4, n_layers=2, max_seq_len=5):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, hidden_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 2,
            dropout=0.1, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x_seq, x_mask):
        h = self.input_proj(x_seq) + self.pos_embed[:, :x_seq.shape[1], :]
        pad_mask = (x_mask == 0)  # True where padded -> ignored by attention
        # guard against fully-padded sequences (shouldn't happen: last position always valid)
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        return h[:, -1, :]  # representation of the current encounter (last position)


class HeterogeneousPatientGAT(nn.Module):
    """Lightweight heterogeneous graph-attention layer over the real bipartite
    patient <-> diagnosis-category graph built in Section 5. Implemented in plain
    PyTorch (no torch_geometric dependency, keeps Colab installs light):
    each patient node attends over a learned embedding table of diagnosis nodes,
    i.e. single-hop attention from the patient's diagnosis neighbor(s)."""
    def __init__(self, tab_input_dim, n_diag_nodes, hidden_dim=64):
        super().__init__()
        self.diag_embed = nn.Embedding(n_diag_nodes, hidden_dim)
        self.patient_proj = nn.Linear(tab_input_dim, hidden_dim)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x_tab, x_diag):
        patient_h = self.patient_proj(x_tab)          # (B, H)
        diag_h = self.diag_embed(x_diag)               # (B, H) - this patient's diagnosis node
        attn_score = torch.sigmoid(self.attn(torch.cat([patient_h, diag_h], dim=-1)))  # (B, 1)
        fused = patient_h + attn_score * diag_h
        return F.relu(self.out_proj(fused))


class HybridModel(nn.Module):
    """Temporal Transformer branch + Heterogeneous Patient GAT branch, fused -> classification head.
    This is the proposed architecture from the project tracker/README."""
    def __init__(self, tab_input_dim, n_diag_nodes, hidden_dim=64, n_heads=4, max_seq_len=5, output_dim=1):
        super().__init__()
        self.temporal_branch = TemporalTransformerBranch(
            tab_input_dim, hidden_dim=hidden_dim, n_heads=n_heads, max_seq_len=max_seq_len
        )
        self.graph_branch = HeterogeneousPatientGAT(tab_input_dim, n_diag_nodes, hidden_dim=hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x_tab, x_seq, x_mask, x_diag):
        h_temporal = self.temporal_branch(x_seq, x_mask)
        h_graph = self.graph_branch(x_tab, x_diag)
        fused = torch.cat([h_temporal, h_graph], dim=-1)
        return self.head(fused)


### Architecture Verification

In [ ]:
input_dim = xb_tab.shape[1]
hybrid = HybridModel(tab_input_dim=input_dim, n_diag_nodes=N_DIAG_NODES, hidden_dim=CONFIG['transformer_hidden_dim'], n_heads=CONFIG['transformer_heads'], max_seq_len=CONFIG['max_seq_len']).to(DEVICE)
print(hybrid)
for name, model in [('hybrid', hybrid)]:
    total_params = sum((p.numel() for p in model.parameters()))
    trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
    print(f'{name}: total params={total_params:,}  trainable={trainable_params:,}  device={next(model.parameters()).device}')


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, patience, ckpt_path, pos_weight=None):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    epoch_bar = tqdm(range(epochs), desc="Training", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        train_loss = 0.0
        batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False, unit="batch")
        for x_tab, x_seq, x_mask, x_diag, yb in batch_bar:
            x_tab, x_seq, x_mask, x_diag, yb = (t.to(DEVICE) for t in (x_tab, x_seq, x_mask, x_diag, yb))
            optimizer.zero_grad()
            preds = model(x_tab, x_seq, x_mask, x_diag).squeeze(-1)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_loss += loss.item() * x_tab.size(0)
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_tab, x_seq, x_mask, x_diag, yb in val_loader:
                x_tab, x_seq, x_mask, x_diag, yb = (t.to(DEVICE) for t in (x_tab, x_seq, x_mask, x_diag, yb))
                preds = model(x_tab, x_seq, x_mask, x_diag).squeeze(-1)
                loss = criterion(preds, yb)
                val_loss += loss.item() * x_tab.size(0)
        val_loss /= len(val_loader.dataset)

        scheduler.step(val_loss)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        epoch_bar.set_postfix(train_loss=f"{train_loss:.4f}", val_loss=f"{val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                epoch_bar.write(f"Early stopping at epoch {epoch+1}")
                break

    return history


In [ ]:
n_pos = train_df[target].sum()
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print('pos_weight (neg/pos ratio, train only):', pos_weight.item())
hybrid_history = train_model(hybrid, train_loader, val_loader, epochs=CONFIG['epochs'], lr=CONFIG['learning_rate'], patience=CONFIG['early_stop_patience'], ckpt_path=os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'), pos_weight=pos_weight)


## 9. Evaluation Metrics

In [ ]:
def get_predictions(model, loader, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))  # reload BEST checkpoint, not last epoch
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_tab, x_seq, x_mask, x_diag, yb in loader:
            x_tab, x_seq, x_mask, x_diag = (t.to(DEVICE) for t in (x_tab, x_seq, x_mask, x_diag))
            logits = model(x_tab, x_seq, x_mask, x_diag).squeeze(-1).cpu().numpy()
            all_preds.append(logits)
            all_targets.append(yb.numpy())
    logits = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    probs = 1 / (1 + np.exp(-logits))
    return probs, targets


def evaluate_classification(probs, targets, threshold=0.5):
    pred_labels = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, pred_labels, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(targets, pred_labels),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "mcc": matthews_corrcoef(targets, pred_labels),
        "roc_auc": roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else None,
        "pr_auc": average_precision_score(targets, probs) if len(np.unique(targets)) > 1 else None,
    }


In [ ]:
results = {}
test_predictions = {}
for name, model, ckpt in [('hybrid', hybrid, os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'))]:
    probs, targets = get_predictions(model, test_loader, ckpt)
    results[name] = evaluate_classification(probs, targets)
    test_predictions[name] = (probs, targets)
print(json.dumps(results, indent=2, default=str))
with open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['val_loss'], label='Hybrid val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Proposed Model Validation Loss')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=300)
plt.show()


In [ ]:
# fig02 - Confusion matrix (hybrid, test set)
hybrid_probs, hybrid_targets = test_predictions["hybrid"]
hybrid_pred_labels = (hybrid_probs >= 0.5).astype(int)

plt.figure(figsize=(5, 4))
cm = confusion_matrix(hybrid_targets, hybrid_pred_labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Hybrid, test set)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_or_scatter.png"), dpi=300)
plt.show()


In [ ]:
# fig03 - ROC / PR curves (hybrid, test set)
fpr, tpr, _ = roc_curve(hybrid_targets, hybrid_probs)
precision, recall, _ = precision_recall_curve(hybrid_targets, hybrid_probs)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC Curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[1].plot(recall, precision)
axes[1].set_title("Precision-Recall Curve"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=300)
plt.show()


### Explainable AI (XAI)
The project methodology relies on attention (Transformer + GAT), so interpretability is grounded in
the model's own learned attention weights rather than a post-hoc surrogate: the diagnosis-attention
gate from `HeterogeneousPatientGAT` (how much each patient's diagnosis-node representation
contributes vs. their own tabular representation) is a direct, faithful explanation signal.

In [ ]:
# fig04 - Feature importance / interpretability: GAT diagnosis-attention weight distribution + top
# tabular features via permutation importance on the trained hybrid model (test set, AUC-based).
hybrid.eval()
with torch.no_grad():
    x_tab_all, x_diag_all = [], []
    for x_tab, x_seq, x_mask, x_diag, yb in test_loader:
        x_tab_all.append(x_tab); x_diag_all.append(x_diag)
    x_tab_all = torch.cat(x_tab_all).to(DEVICE)
    x_diag_all = torch.cat(x_diag_all).to(DEVICE)

    patient_h = hybrid.graph_branch.patient_proj(x_tab_all)
    diag_h = hybrid.graph_branch.diag_embed(x_diag_all)
    attn_scores = torch.sigmoid(hybrid.graph_branch.attn(torch.cat([patient_h, diag_h], dim=-1))).cpu().numpy().ravel()

plt.figure(figsize=(6, 4))
sns.histplot(attn_scores, bins=30)
plt.title("GAT diagnosis-node attention weight distribution (test set)")
plt.xlabel("Attention weight on diagnosis-node embedding")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=300)
plt.show()
print("Mean attention on diagnosis node:", attn_scores.mean(), "  std:", attn_scores.std())


### Error Analysis

In [ ]:
# fig05 - Error analysis: false negatives (missed early readmissions) by diagnosis category
fn_mask = (hybrid_targets == 1) & (hybrid_pred_labels == 0)
fp_mask = (hybrid_targets == 0) & (hybrid_pred_labels == 1)
print(f"False negatives (missed early readmissions): {fn_mask.sum()} / {hybrid_targets.sum()} positives")
print(f"False positives: {fp_mask.sum()} / {(hybrid_targets==0).sum()} negatives")

test_diag_cats = test_df["diag_1_cat"].values
fn_diag_counts = pd.Series(test_diag_cats[fn_mask]).value_counts().head(10)

plt.figure(figsize=(8, 5))
fn_diag_counts.plot(kind="barh")
plt.title("Top diagnosis categories among missed early readmissions (false negatives)")
plt.xlabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=300)
plt.show()


### Computational Efficiency

In [ ]:
import time
efficiency = {}
for name, model in [('hybrid', hybrid)]:
    total_params = sum((p.numel() for p in model.parameters()))
    trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
    model.eval()
    x_tab_b, x_seq_b, x_mask_b, x_diag_b, _ = next(iter(test_loader))
    x_tab_b, x_seq_b, x_mask_b, x_diag_b = (t.to(DEVICE) for t in (x_tab_b, x_seq_b, x_mask_b, x_diag_b))
    with torch.no_grad():
        for _ in range(3):
            model(x_tab_b, x_seq_b, x_mask_b, x_diag_b)
        start = time.time()
        n_runs = 20
        for _ in range(n_runs):
            model(x_tab_b, x_seq_b, x_mask_b, x_diag_b)
        elapsed = (time.time() - start) / n_runs
    efficiency[name] = {'total_params': total_params, 'trainable_params': trainable_params, 'avg_batch_inference_time_sec': elapsed, 'throughput_samples_per_sec': x_tab_b.shape[0] / elapsed}
print(json.dumps(efficiency, indent=2))
with open(os.path.join(CONFIG['results_dir'], 'efficiency.json'), 'w') as f:
    json.dump(efficiency, f, indent=2)
